# CITE-seq BMMC scIB benchmark

In [ ]:
mode = "subset"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
    "#000000",
    "#ffcc00",
    "#8b4513",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
    "schwarz",
    "goldgelb",
    "sattelbraun",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import sys
import os

_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "figS1_S12_neurips2021_cite_bmmc_scib")
os.makedirs(_OUTDIR_ABS, exist_ok=True)

sys.path.append(os.path.abspath("../.experiment_data/primary/benchmarking"))

import contextlib
import functools

_POSTPROC_CWD = os.getcwd()

@contextlib.contextmanager
def _in_primary_dir():
    os.chdir(os.path.join(_POSTPROC_CWD, "../.experiment_data/primary"))
    try:
        yield
    finally:
        os.chdir(_POSTPROC_CWD)

with _in_primary_dir():
    import muon as _muon
    _orig_sample_obs = _muon.pp.sample_obs
    def _patched_sample_obs(data, *a, **kw):
        return _orig_sample_obs(data, *a, **kw).copy()
    _muon.pp.sample_obs = _patched_sample_obs
    import datasets

for _name in dir(datasets):
    if _name.startswith("load_"):
        def _make_wrapper(fn):
            @functools.wraps(fn)
            def _wrapper(*a, **kw):
                with _in_primary_dir():
                    return fn(*a, **kw)
            return _wrapper
        setattr(datasets, _name, _make_wrapper(getattr(datasets, _name)))

import glob
import pandas as pd
import scipy
import numpy as np
import os
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import glob
from collections import Counter
from pathlib import Path
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
import glob
import pandas as pd
import scipy
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
import textwrap
from adjustText import adjust_text
import matplotlib.patches as mpatches
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


In [ ]:
def map_cell_type_to_family(cell_type):
    cell_type = cell_type.lower()

    if any(x in cell_type for x in ["hsc", "prog"]):
        return "Progenitor"
    elif "b" in cell_type:
        return "B cell"
    elif any(x in cell_type for x in ["nk", "ilc"]):
        return "ILC/NK"
    elif any(x in cell_type for x in ["cd4", "cd8", "treg", "dnT", "gdT", "mait"]):
        return "T cell"
    elif "mono" in cell_type or "mac" in cell_type:
        return "Monocyte"
    elif "dc" in cell_type:
        return "Dendritic cell"
    elif any(x in cell_type for x in ["erythro", "nomo", "retic", "normo"]):
        return "Erythroid"
    elif any(x in cell_type for x in ["plasmablast", "plasma"]):
        return "B cell"
    elif "platelet" in cell_type:
        return "Platelet"
    elif "doublet" in cell_type:
        return "Artifact"
    else:
        return "Other"

family_colors = {
    "Progenitor": "#4CAF50",
    "B cell": "#E53935",
    "ILC/NK": "#FB8C00",
    "T cell": "#FDD835",
    "Monocyte": "#F06292",
    "Dendritic cell": "#8E24AA",
    "Erythroid": "#6D4C41",
    "Platelet": "#90A4AE",
    "Artifact": "#BDBDBD",
    "Other": "#9E9E9E",
}

from matplotlib.colors import to_rgb, to_hex
import colorsys

def generate_family_shades(family_labels, base_color):
    '''Generate different shades for subtypes in a given family.'''
    base_rgb = to_rgb(base_color)
    hsv = colorsys.rgb_to_hsv(*base_rgb)
    n = len(family_labels)
    shades = []

    for i in range(n):
        value = 0.6 + (0.4 * (i / max(n - 1, 1)))
        rgb = colorsys.hsv_to_rgb(hsv[0], hsv[1], value)
        shades.append(to_hex(rgb))

    return dict(zip(family_labels, shades))

def generate_family_shades(family_labels, base_color, hue_shift=0.08):
    '''Generate visually distinct colors within a family by shifting hue slightly.'''
    base_rgb = to_rgb(base_color)
    h, s, v = colorsys.rgb_to_hsv(*base_rgb)
    n = len(family_labels)
    shades = []

    for i in range(n):
        new_h = (h + hue_shift * (i / max(n - 1, 1)) - hue_shift / 2) % 1.0
        new_s = min(max(s * (0.8 + 0.2 * (i / max(n - 1, 1))), 0), 1)
        new_v = min(max(v * (0.8 + 0.2 * (i / max(n - 1, 1))), 0), 1)
        rgb = colorsys.hsv_to_rgb(new_h, new_s, new_v)
        shades.append(to_hex(rgb))

    return dict(zip(family_labels, shades))

from collections import defaultdict

def build_color_map(cell_types):
    family_to_types = defaultdict(list)
    for ct in cell_types:
        family = map_cell_type_to_family(ct)
        family_to_types[family].append(ct)

    type_to_color = {}
    for family, types in family_to_types.items():
        base_color = family_colors.get(family, "#9E9E9E")
        shades = generate_family_shades(types, base_color)
        type_to_color.update(shades)

    return type_to_color


In [ ]:
def show_umap_plot(X_umap, labels, cmap, encode_first_as_grey=False, continuous=False, title='', figsize=(3, 3), save_name=False):
    set_figure_params(4, 4)
    if not continuous:
        unique_labels, encoded_labels = np.unique(labels, return_inverse=True)
        n_categories = len(unique_labels)
        base_palette = sns.color_palette('husl', n_categories)
        if n_categories > 10:
            reordered_palette = []
            spacing = 6
            for i in range(len(base_palette)):
                index = i // spacing + i % spacing * (len(base_palette) // spacing)
                reordered_palette.append(base_palette[index % n_categories])
        else:
            reordered_palette = base_palette
        if not encode_first_as_grey:
            new_palette = reordered_palette
            cmap = ListedColormap(new_palette)
        else:
            new_palette = ['#eff1f2'] + reordered_palette[1:]
            cmap = ListedColormap(new_palette)
    else:
        encoded_labels = labels
    fig, ax = plt.subplots()
    ax.set_aspect('equal', 'box')
    if continuous:
        sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=0.1, cmap=cmap, c=encoded_labels, edgecolor='none', rasterized=True)
    else:
        sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=0.1, c=encoded_labels, cmap=cmap, edgecolor='none', rasterized=True)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_box_aspect(1)
    plt.title(title)
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()
    plt.close()

def show_umap_plot(X_umap, labels, cmap, encode_first_as_grey=False, continuous=False, title='', figsize=(3, 3), save_name=False):
    set_figure_params(4, 4)
    unique_labels = np.unique(labels)
    label_to_int = {label: i for i, label in enumerate(unique_labels)}
    encoded_labels = [label_to_int[l] for l in labels]
    colors = [type_to_color.get(label, '#9E9E9E') for label in unique_labels]
    cmap = ListedColormap(colors)
    fig, ax = plt.subplots()
    ax.set_aspect('equal', 'box')
    sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=0.1, c=encoded_labels, cmap=cmap, edgecolor='none', rasterized=True)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_box_aspect(1)
    plt.title(title)
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()
    plt.close()

def show_colorbar(cmap, save_name):
    fig, ax = plt.subplots(figsize=(1.5, 4))
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=1))
    cbar = plt.colorbar(sm, ax=ax)
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label('GO activation', rotation=270, labelpad=15)
    plt.axis('off')
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()

def show_legend(labels, save_name):
    unique_labels, encoded_labels = np.unique(labels, return_inverse=True)
    n_categories = len(unique_labels)
    base_palette = sns.color_palette('husl', n_categories)
    if n_categories > 10:
        reordered_palette = []
        spacing = 6
        for i in range(len(base_palette)):
            index = i // spacing + i % spacing * (len(base_palette) // spacing)
            reordered_palette.append(base_palette[index % n_categories])
    else:
        reordered_palette = base_palette
    new_palette = reordered_palette
    cmap = ListedColormap(new_palette)
    fig, ax = plt.subplots(figsize=(2, 8))
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=cmap(i)) for i in range(n_categories)]
    ax.legend(handles, unique_labels, bbox_to_anchor=(1, 1), loc='upper left')
    plt.axis('off')
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()

def show_legend(labels, save_name=False):
    unique_labels = sorted(set(labels))
    colors = [type_to_color.get(label, '#9E9E9E') for label in unique_labels]
    fig, ax = plt.subplots(figsize=(2, min(0.3 * len(unique_labels), 15)))
    handles = [plt.Line2D([0], [0], marker='o', linestyle='None', markersize=6, markerfacecolor=color, markeredgecolor='k') for color in colors]
    ax.legend(handles, unique_labels, bbox_to_anchor=(1, 1), loc='upper left', frameon=False)
    plt.axis('off')
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()
    plt.close()

def show_legend(labels, save_name=False):
    family_to_labels = defaultdict(list)
    for label in np.unique(labels):
        family = map_cell_type_to_family(label)
        family_to_labels[family].append(label)
    family_order = list(family_colors.keys())
    handles = []
    legend_labels = []
    for family in family_order:
        for label in sorted(family_to_labels.get(family, [])):
            color = type_to_color.get(label, '#9E9E9E')
            handles.append(plt.Line2D([0], [0], marker='o', linestyle='None', markersize=6, markerfacecolor=color, markeredgecolor='k'))
            legend_labels.append(f'{label}')
    fig, ax = plt.subplots()
    ax.legend(handles, legend_labels, bbox_to_anchor=(1, 1), loc='upper left', frameon=False)
    plt.axis('off')
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()
    plt.close()


In [ ]:
def generate_results_tables(results, save_name=None):
    for hyper_label, hyper_bool in zip(['default', 'hyper'], [False, True]):
        results_filtered = results[(results['scib_type'] == 'combined') & (results['hyper'] == False)]
        results_table = results_filtered.pivot(index='tool_label', columns='paired_rate', values='scib')
        results_table = results_table[sorted(results_table.columns, reverse=True)]
        unique_elements, indices = np.unique(results_filtered['tool_label'], return_index=True)
        custom_index_order = unique_elements[np.argsort(indices)]
        results_table = results_table.reindex(custom_index_order)
        results_table.to_csv(save_name + f'_{hyper_label}_results.table.csv')

def generate_results_plot(results, paired_rates, tools_labels):
    results_plot = []
    for paired_rate in paired_rates:
        print(paired_rate)
        results_plot_sub = []
        for tool_label in tools_labels:
            if tool_label == 'WNN' or tool_label == 'MIDAS' or tool_label == 'MOFA+':
                hyper_bool = False
            else:
                hyper_bool = True
            print(results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label)])
            print(paired_rate)
            print(tool_label)
            result_plot_hyper_label = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'label_adj') & (results['hyper'] == hyper_bool)]['scib']
            result_plot_hyper_label = result_plot_hyper_label.iloc[0] if len(result_plot_hyper_label) > 0 else 0
            result_plot_hyper_modality = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'modality_adj') & (results['hyper'] == hyper_bool)]['scib']
            result_plot_hyper_modality = result_plot_hyper_modality.iloc[0] if len(result_plot_hyper_modality) > 0 else 0
            result_plot_hyper_batch = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'batch_adj') & (results['hyper'] == hyper_bool)]['scib']
            result_plot_hyper_batch = result_plot_hyper_batch.iloc[0] if len(result_plot_hyper_batch) > 0 else 0
            print(np.array([result_plot_hyper_label, result_plot_hyper_modality, result_plot_hyper_batch]))
            results_plot_sub.append(np.array([result_plot_hyper_label, result_plot_hyper_modality, result_plot_hyper_batch]))
        results_plot.append(np.array(results_plot_sub))
    return results_plot

def generate_results_plot_old(results, paired_rates, tools_labels):
    results_plot = []
    for paired_rate in paired_rates:
        print(paired_rate)
        results_plot_sub = []
        for tool_label in tools_labels:
            print(results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label)])
            print(paired_rate)
            print(tool_label)
            result_plot_label = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'label_adj') & (results['hyper'] == False)]['scib']
            result_plot_label = result_plot_label.iloc[0] if len(result_plot_label) > 0 else 0
            result_plot_hyper_label = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'label_adj') & (results['hyper'] == True)]['scib']
            result_plot_hyper_label = result_plot_hyper_label.iloc[0] if len(result_plot_hyper_label) > 0 else 0
            result_plot_hyper_label = max(0, result_plot_hyper_label - result_plot_label)
            result_plot_modality = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'modality_adj') & (results['hyper'] == False)]['scib']
            result_plot_modality = result_plot_modality.iloc[0] if len(result_plot_modality) > 0 else 0
            result_plot_hyper_modality = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'modality_adj') & (results['hyper'] == True)]['scib']
            result_plot_hyper_modality = result_plot_hyper_modality.iloc[0] if len(result_plot_hyper_modality) > 0 else 0
            result_plot_hyper_modality = max(0, result_plot_hyper_modality - result_plot_modality)
            result_plot_batch = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'batch_adj') & (results['hyper'] == False)]['scib']
            result_plot_batch = result_plot_batch.iloc[0] if len(result_plot_batch) > 0 else 0
            result_plot_hyper_batch = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'batch_adj') & (results['hyper'] == True)]['scib']
            result_plot_hyper_batch = result_plot_hyper_batch.iloc[0] if len(result_plot_hyper_batch) > 0 else 0
            result_plot_hyper_batch = max(0, result_plot_hyper_batch - result_plot_batch)
            print(np.array([result_plot_label, result_plot_modality, result_plot_batch, result_plot_hyper_label, result_plot_hyper_modality, result_plot_hyper_batch]))
            results_plot_sub.append(np.array([result_plot_label, result_plot_modality, result_plot_batch, result_plot_hyper_label, result_plot_hyper_modality, result_plot_hyper_batch]))
        results_plot.append(np.array(results_plot_sub))
    return results_plot

def plot_performance_old(tools_labels, results, results_plot, bar_width=0.05, bar_spacing=0, figsize=(14, 7), show_legend=True, save_name=False):
    set_figure_params('single', 2.5)

    def generate_patterns(n_parts):
        if n_parts == 4:
            return ['//', None, '//', None]
        elif n_parts == 6:
            return ['//', 'oo', '..', '//', 'oo', '..']

    def lighten_color(color, amount=0.6):
        c = np.array(mcolors.to_rgba(color))
        white = np.array([1, 1, 1, 1])
        return tuple((1 - amount) * c + amount * white)
    groups = [len(tools_labels)] + [len(tools_labels) - 2] * 4
    n_parts = 6
    patterns = generate_patterns(n_parts)
    data = results_plot
    base_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5', '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5']
    base_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    light_colors = [lighten_color(color, amount=0.4) for color in base_colors]
    fig, ax = plt.subplots()
    bar_width = bar_width
    indices = np.arange(len(paired_rates))
    group_centers = []
    for i, group_data in enumerate(data):
        group_center = indices[i] + (bar_width + bar_spacing) * (groups[i] - 1) / 2
        group_centers.append(group_center)
        for j in range(groups[i]):
            bottom = np.zeros(group_data.shape[1])
            for k in range(group_data.shape[1]):
                color = base_colors[j] if k < n_parts // 2 else light_colors[j]
                bar_position = indices[i] + j * (bar_width + bar_spacing)
                ax.bar(bar_position, group_data[j, k], bar_width, bottom=bottom, color=color, capsize=3, edgecolor='black', hatch=patterns[k])
                bottom += group_data[j, k]
    ax.set_xticks(group_centers)
    ax.set_xticklabels([f'{pr:.1f}' for pr in paired_rates])
    ax.set_xlabel('paired rate')
    ax.set_ylabel('combined scib')
    ax.set_ylim(bottom=0, top=max(results[results['scib_type'] == 'combined']['scib']) + 0.05)
    legend_handles_tools = [plt.Rectangle((0, 0), 1, 1, color=color, edgecolor='black') for color in base_colors]
    if show_legend:
        ax.legend(legend_handles_tools, tools_labels, loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()

def plot_performance(tools_labels, results, results_plot, bar_width=0.05, bar_spacing=0, figsize=(14, 7), show_legend=True, save_name=False):
    set_figure_params('single', 2)

    def lighten_color(color, amount=0.6):
        c = np.array(mcolors.to_rgba(color))
        white = np.array([1, 1, 1, 1])
        return tuple((1 - amount) * c + amount * white)
    groups = [len(tools_labels)] + [len(tools_labels) - 4] * 4
    data = results_plot
    base_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5', '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5']
    biological_preseveration_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    batch_colors = [lighten_color(color, amount=0.6) for color in biological_preseveration_colors]
    modality_colors = [lighten_color(color, amount=0.3) for color in biological_preseveration_colors]
    colors = [biological_preseveration_colors, modality_colors, batch_colors]
    fig, ax = plt.subplots()
    bar_width = bar_width
    indices = np.arange(len(paired_rates))
    group_centers = []
    for i, group_data in enumerate(data):
        group_center = indices[i] + (bar_width + bar_spacing) * (groups[i] - 1) / 2
        group_centers.append(group_center)
        for j in range(groups[i]):
            bottom = np.zeros(group_data.shape[1])
            for k in range(group_data.shape[1]):
                color = colors[k][j]
                bar_position = indices[i] + j * (bar_width + bar_spacing)
                ax.bar(bar_position, group_data[j, k], bar_width, bottom=bottom, color=color, capsize=3, edgecolor='black')
                bottom += group_data[j, k]
    ax.set_xticks(group_centers)
    ax.set_xticklabels([f'{pr:.1f}' for pr in paired_rates])
    ax.set_xlabel('paired rate')
    ax.set_ylabel('combined scib')
    ax.set_ylim(bottom=0, top=max(results[results['scib_type'] == 'combined']['scib']) + 0.05)
    legend_handles_tools = [plt.Rectangle((0, 0), 1, 1, color=color, edgecolor='black') for color in biological_preseveration_colors[:len(tools_labels)]]
    scib_legend_handles_tools = [plt.Rectangle((0, 0), 1, 1, color='white', edgecolor='white')] + [plt.Rectangle((0, 0), 1, 1, color=lighten_color(biological_preseveration_colors[0], amount=lighten_amount), edgecolor='black') for lighten_amount in [0, 0.3, 0.6]]
    scib_labels = ['', 'biological preservation', 'modality alignment', 'batch correction']
    if show_legend:
        ax.legend(legend_handles_tools + scib_legend_handles_tools, tools_labels + scib_labels, loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()


In [ ]:
def get_best_hyper(path):

    num_hyper = sum(
        os.path.isdir(os.path.join(path, "hyper", entry))
        for entry in os.listdir(os.path.join(path, "hyper"))
    )
    scib_hyper_labels = list(
        glob.glob(os.path.join(path, "hyper", "hyper*", "scib_results_label.csv"))
    )
    scib_hyper_label = np.array(
        [
            float(pd.read_csv(path_label)["Bio conservation"].iloc[0])
            for path_label in scib_hyper_labels
        ]
    )

    if len(scib_hyper_label) > 0:
        if paired_rate != 1 and paired_rate != 1.0:
            scib_hyper_modality = np.array(
                [
                    float(pd.read_csv(path_label)["Batch correction"].iloc[0])
                    for path_label in glob.glob(
                        os.path.join(path, "hyper", "hyper*", "scib_results_modality.csv")
                    )
                ]
            )
        else:
            scib_hyper_modality = [None] * len(scib_hyper_labels)
        if batch_key:
            scib_hyper_batch = np.array(
                [
                    float(pd.read_csv(path_label)["Batch correction"].iloc[0])
                    for path_label in glob.glob(
                        os.path.join(path, "hyper", "hyper*", f"scib_results_{batch_key}.csv")
                    )
                ]
            )
        else:
            scib_hyper_batch = [None] * len(scib_hyper_labels)

        if batch_key:
            if paired_rate != 1 and paired_rate != 1.0:
                scib_hyper = (
                    0.6 * scib_hyper_label + 0.2 * scib_hyper_modality + 0.2 * scib_hyper_batch
                )
            else:
                scib_hyper = 0.6 * scib_hyper_label + 0.4 * scib_hyper_batch
        else:
            if paired_rate != 1 and paired_rate != 1.0:
                scib_hyper = 0.6 * scib_hyper_label + 0.4 * scib_hyper_modality
            else:
                scib_hyper = scib_hyper_label
        pos_scib_hyper = np.where(scib_hyper == scib_hyper.max())[0][0]
        path_scib_hyper = str(
            Path(
                glob.glob(os.path.join(path, "hyper", "hyper*", "scib_results_label.csv"))[
                    pos_scib_hyper
                ]
            ).parent
        )
    else:
        print("Issue with path")
        num_hyper = None
        path_scib_hyper = None
        scib_hyper = None

    return num_hyper, path_scib_hyper, scib_hyper.max()


In [ ]:
def get_scib_results(path_default_hyper, results, batch_key, tool_label="", hyper=False):
    _visualize = paired_rate == 1 or paired_rate == 1.0
    X_umap = np.load(os.path.join(path_default_hyper, "X_umap.npy"))
    try:
        labels = np.load(os.path.join(path_default_hyper, "labels.npy"), allow_pickle=True)[
            train_indices
        ]
    except IndexError:
        labels = np.load(os.path.join(path_default_hyper, "labels.npy"), allow_pickle=True)
    show_umap_plot(
        X_umap,
        labels,
        cmap,
        continuous=False,
        save_name=(f"{_OUTDIR_ABS}/{dataset_name}_labels_{tool_label}_{paired_rate}" if _visualize else False),
    )
    scib_label = float(
        pd.read_csv(os.path.join(path_default_hyper, "scib_results_label.csv"))[
            "Bio conservation"
        ].iloc[0]
    )
    results = pd.concat(
        [
            results,
            pd.DataFrame(
                [[scib_label, "label", tool_label, paired_rate, hyper]],
                columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
            ),
        ],
        ignore_index=True,
    )

    if paired_rate != 1 and paired_rate != 1.0:
        try:
            modality = np.load(os.path.join(path_default_hyper, "modality.npy"), allow_pickle=True)[
                train_indices
            ]
        except IndexError:
            modality = np.load(os.path.join(path_default_hyper, "modality.npy"), allow_pickle=True)

        if "sparsevi" in tool_label.lower():
            X_umap_expression = np.load(os.path.join(path_default_hyper, "X_umap_expression.npy"))
            show_umap_plot(
                X_umap_expression,
                labels,
                cmap,
                continuous=False,
                save_name=(f"{_OUTDIR_ABS}/{dataset_name}_labels_{tool_label}_{paired_rate}_expression" if _visualize else False),
            )
            show_umap_plot(
                X_umap_expression,
                modality,
                cmap,
                continuous=False,
                save_name=(f"{_OUTDIR_ABS}/{dataset_name}_modality_{tool_label}_{paired_rate}_expression" if _visualize else False),
            )
            X_umap_protein = np.load(os.path.join(path_default_hyper, "X_umap_protein.npy"))
            show_umap_plot(
                X_umap_protein,
                labels,
                cmap,
                continuous=False,
                save_name=(f"{_OUTDIR_ABS}/{dataset_name}_labels_{tool_label}_{paired_rate}_protein" if _visualize else False),
            )
            show_umap_plot(
                X_umap_protein,
                modality,
                cmap,
                continuous=False,
                save_name=(f"{_OUTDIR_ABS}/{dataset_name}_modality_{tool_label}_{paired_rate}_protein" if _visualize else False),
            )

        show_umap_plot(
            X_umap,
            modality,
            cmap,
            continuous=False,
            save_name=(f"{_OUTDIR_ABS}/{dataset_name}_modality_{tool_label}_{paired_rate}" if _visualize else False),
        )
        scib_modality = float(
            pd.read_csv(os.path.join(path_default_hyper, "scib_results_modality.csv"))[
                "Batch correction"
            ].iloc[0]
        )
        results = pd.concat(
            [
                results,
                pd.DataFrame(
                    [[scib_modality, "modality", tool_label, paired_rate, hyper]],
                    columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                ),
            ],
            ignore_index=True,
        )

    if batch_key:
        try:
            batch = np.load(os.path.join(path_default_hyper, f"batch.npy"), allow_pickle=True)[
                train_indices
            ]
        except IndexError:
            batch = np.load(os.path.join(path_default_hyper, f"batch.npy"), allow_pickle=True)

        if "sparsevi" in tool_label.lower():
            if paired_rate != 1 and paired_rate != 1.0:
                show_umap_plot(
                    X_umap_expression,
                    batch,
                    cmap,
                    continuous=False,
                    save_name=(f"{_OUTDIR_ABS}/{dataset_name}_batch_{tool_label}_{paired_rate}_expression" if _visualize else False),
                )
                show_umap_plot(
                    X_umap_protein,
                    batch,
                    cmap,
                    continuous=False,
                    save_name=(f"{_OUTDIR_ABS}/{dataset_name}_batch_{tool_label}_{paired_rate}_protein" if _visualize else False),
                )

        show_umap_plot(
            X_umap,
            batch,
            cmap,
            continuous=False,
            save_name=(f"{_OUTDIR_ABS}/{dataset_name}_{batch_key}_{tool_label}_{paired_rate}" if _visualize else False),
        )
        scib_batch = float(
            pd.read_csv(os.path.join(path_default_hyper, f"scib_results_{batch_key}.csv"))[
                "Batch correction"
            ].iloc[0]
        )
        results = pd.concat(
            [
                results,
                pd.DataFrame(
                    [[scib_batch, "batch", tool_label, paired_rate, hyper]],
                    columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                ),
            ],
            ignore_index=True,
        )
        if paired_rate != 1 and paired_rate != 1.0:
            scib = 0.6 * scib_label + 0.2 * scib_modality + 0.2 * scib_batch
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_label * 0.6, "label_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_modality * 0.2, "modality_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_batch * 0.2, "batch_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )

        else:
            scib = 0.6 * scib_label + 0.4 * scib_batch
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_label * 0.6, "label_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_batch * 0.4, "batch_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )
    else:
        if paired_rate != 1 and paired_rate != 1.0:
            scib = 0.6 * scib_label + 0.4 * scib_modality
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_label * 0.6, "label_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_modality * 0.4, "modality_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )
        else:
            scib = scib_label
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_label, "label_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )
    results = pd.concat(
        [
            results,
            pd.DataFrame(
                [[scib, "combined", tool_label, paired_rate, hyper]],
                columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
            ),
        ],
        ignore_index=True,
    )

    return results


In [ ]:
import sys
import os

sys.path.append(os.path.abspath("../.experiment_data/primary/benchmarking"))
from metrics import calculate_scib_metrics
import anndata as ad

n_bootstrap = 2

def calculate_scib_bootstrapped(
    path_scib_hyper, results, tool_label="", paired_rate=None, hyper=True
):

    if hyper:
        X_latent_addix = path_scib_hyper.split("/")[-3].split("_")[0]
    else:
        X_latent_addix = path_scib_hyper.split("/")[-2].split("_")[0]
    X_latent = np.load(os.path.join(path_scib_hyper, f"X_{X_latent_addix}.npy"))
    try:
        labels = np.load(os.path.join(path_scib_hyper, "labels.npy"), allow_pickle=True)[
            train_indices
        ]
    except IndexError:
        labels = np.load(os.path.join(path_scib_hyper, "labels.npy"), allow_pickle=True)

    if paired_rate != 1 and paired_rate != 1.0:
        try:
            modality = np.load(os.path.join(path_scib_hyper, "modality.npy"), allow_pickle=True)[
                train_indices
            ]
        except IndexError:
            modality = np.load(os.path.join(path_scib_hyper, "modality.npy"), allow_pickle=True)
    else:
        modality = np.zeros_like(labels)

    if batch_key:
        try:
            batch = np.load(os.path.join(path_scib_hyper, f"batch.npy"), allow_pickle=True)[
                train_indices
            ]
        except IndexError:
            batch = np.load(os.path.join(path_scib_hyper, f"batch.npy"), allow_pickle=True)
    else:
        batch = np.zeros_like(labels)

    scib_bootstrapped = []
    for _ in range(n_bootstrap):
        indices = np.random.choice(len(labels), size=int(0.95 * len(labels)), replace=False)
        X_latent_bootstrapped = X_latent[indices]
        fake_anndata = ad.AnnData(
            X=X_latent_bootstrapped,
            obs=pd.DataFrame(
                data={
                    labels_key: labels[indices],
                    "modality": modality[indices],
                    batch_key: batch[indices],
                }
            ),
        )
        scib_bootstrapped.append(
            calculate_scib_metrics(
                None,
                X_latent_bootstrapped,
                fake_anndata,
                labels_key,
                modality_key="modality" if paired_rate != 1 and paired_rate != 1.0 else None,
                batch_key=batch_key,
            )
        )

    np.percentile(scib_bootstrapped, 2.5)
    np.percentile(scib_bootstrapped, 97.5)
    results = pd.concat(
        [
            results,
            pd.DataFrame(
                [[scib_bootstrapped, "combined_scib", tool_label, paired_rate, hyper]],
                columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
            ),
        ],
        ignore_index=True,
    )
    return results


In [ ]:
(
    mdata,
    mdata_counts,
    adata,
    train_indices,
    val_indices,
    n_genes,
    n_regions,
    n_snps,
    labels_key,
    further_labels_keys,
    batch_key,
    patient_key,
    further_batch_keys,
    further_continuous_batch_keys,
    n_patient_covariates,
    protein_expression_obsm_key,
) = datasets.load_neurips2021_cite_BMMC(1.0, True, True)


In [ ]:
all_cell_types = list(np.unique(mdata.mod["rna"].obs[labels_key]))
type_to_color = build_color_map(all_cell_types)


In [ ]:
wnn_paths = [
    "../.experiment_data/primary/experiments/wnn_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_False_s42/",
    None,
    None,
    None,
    None,
]
midas_ec_paths = [
    "../.experiment_data/primary/experiments/midas_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_False_batcov_False_patcov_False_s0/",
    "../.experiment_data/primary/experiments/midas_neurips2021_cite_bmmc_0.8_scib_hvar_True_batscib_False_patscib_False_scarches_False_ec_True_batcov_True_patcov_False_s0/",
    "../.experiment_data/primary/experiments/midas_neurips2021_cite_bmmc_0.6_scib_hvar_True_batscib_False_patscib_False_scarches_False_ec_True_batcov_True_patcov_False_s0/",
    "../.experiment_data/primary/experiments/midas_neurips2021_cite_bmmc_0.4_scib_hvar_True_batscib_False_patscib_False_scarches_False_ec_True_batcov_True_patcov_False_s0/",
    "../.experiment_data/primary/experiments/midas_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_False_patscib_False_scarches_False_ec_True_batcov_True_patcov_False_s0/",
]
mofa_paths = [
    "../.experiment_data/primary/experiments/mofa_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_False_s0/",
    None,
    None,
    None,
    None,
]
multimil_batscib_condition_encoders_paths = [
    "../.experiment_data/primary/experiments/multimil_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ce_True_batcov_True_patcov_False_s0/",
    "../.experiment_data/primary/experiments/multimil_neurips2021_cite_bmmc_0.8_scib_hvar_True_batscib_True_patscib_False_scarches_False_ce_True_batcov_True_patcov_False_s0/",
    "../.experiment_data/primary/experiments/multimil_neurips2021_cite_bmmc_0.6_scib_hvar_True_batscib_True_patscib_False_scarches_False_ce_True_batcov_True_patcov_False_s0/",
    "../.experiment_data/primary/experiments/multimil_neurips2021_cite_bmmc_0.4_scib_hvar_True_batscib_True_patscib_False_scarches_False_ce_True_batcov_True_patcov_False_s0/",
    "../.experiment_data/primary/experiments/multimil_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_True_patscib_False_scarches_False_ce_True_batcov_True_patcov_False_s0/",
]
multivi_deeply_inject_covariates_paths = [
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_dense_s0/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.8_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_dense_s0/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.6_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_dense_s0/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.4_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_dense_s0/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_dense_s0/",
]
sparsevi_deeply_inject_covariates_paths = [
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.8_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.6_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.4_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
]
sparsevi_deeply_inject_covariates_rigl_dynamic_paths = [
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_rigl_dynamic_s0/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.8_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_rigl_dynamic_s0/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.6_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_rigl_dynamic_s0/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.4_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_rigl_dynamic_s0/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_rigl_dynamic_s0/",
]
sparsevi_v2_moe_deeply_inject_covariates_paths = [
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_0.8_scib_hvar_True_batscib_True_patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_0.6_scib_hvar_True_batscib_True_patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_0.4_scib_hvar_True_batscib_True_patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_True_patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
]
sparsevi_v2_deeply_inject_covariates_paths = [
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    None,
    None,
    None,
    None,
]
sparsevi_v2_deeply_inject_covariates_rigl_dynamic_paths = [
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_rigl_dynamic_10_s0/",
    None,
    None,
    None,
    None,
]
totalvi_paths = [
    "../.experiment_data/primary/experiments/totalvi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_batcov_False_patcov_False_s0/",
    None,
    None,
    None,
    None,
]
sparsevi_deeply_inject_covariates_paths_combined = [
    [
        "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
        "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    ],
    [
        "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.8_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
        "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_0.8_scib_hvar_True_batscib_True_patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    ],
    [
        "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.6_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
        "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_0.6_scib_hvar_True_batscib_True_patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    ],
    [
        "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.4_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
        "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_0.4_scib_hvar_True_batscib_True_patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    ],
    [
        "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
        "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_True_patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    ],
]


In [ ]:
dataset_name = "neurips2021_cite_bmmc"
paired_rates = [1.0, 0.8, 0.6, 0.4, 0.2]

results = pd.DataFrame(columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"])

tools_labels = [
    "SparseVI",
    "MultiVI",
    "MultiMIL",
    "MIDAS",
    "WNN",
    "MOFA+",
    "TotalVI",
]
tools_hyper = [
    True,
    True,
    True,
    False,
    False,
    False,
    True,
]
tools_paths = [
    sparsevi_deeply_inject_covariates_paths_combined,
    multivi_deeply_inject_covariates_paths,
    multimil_batscib_condition_encoders_paths,
    midas_ec_paths,
    wnn_paths,
    mofa_paths,
    totalvi_paths,
]

for tool_label, tool_hyper, tool_paths in tqdm(zip(tools_labels, tools_hyper, tools_paths)):
    for paired_rate, path in zip(paired_rates, tool_paths):
        if path is not None:
            if type(path) == list:
                results = get_scib_results(
                    os.path.join(path[0], "default"), results, batch_key, tool_label, hyper=False
                )
            else:
                results = get_scib_results(
                    os.path.join(path, "default"), results, batch_key, tool_label, hyper=False
                )
            if tool_hyper:
                print(path)
                if type(path) == list:
                    num_hyper_1, path_scib_hyper_1, scib_max1 = get_best_hyper(path[0])
                    print(num_hyper_1)
                    num_hyper_2, path_scib_hyper_2, scib_max2 = get_best_hyper(path[1])
                    print(num_hyper_2)
                    path_scib_hyper = (
                        path_scib_hyper_1 if scib_max1 > scib_max2 else path_scib_hyper_2
                    )
                else:
                    num_hyper, path_scib_hyper, scib_max = get_best_hyper(path)
                    print(num_hyper)
                if path_scib_hyper is not None:
                    results = get_scib_results(
                        path_scib_hyper, results, batch_key, tool_label, hyper=True
                    )


In [ ]:
dataset_name = "neurips2021_cite_bmmc"
paired_rates = [1.0, 0.8, 0.6, 0.4, 0.2]

results = pd.DataFrame(columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"])

tools_labels = [
    "SparseVI",
    "MultiVI",
    "MultiMIL",
    "MIDAS",
    "WNN",
    "MOFA+",
    "TotalVI",
]
tools_hyper = [
    True,
    True,
    True,
    False,
    False,
    False,
    True,
]
tools_paths = [
    sparsevi_deeply_inject_covariates_paths_combined,
    multivi_deeply_inject_covariates_paths,
    multimil_batscib_condition_encoders_paths,
    midas_ec_paths,
    wnn_paths,
    mofa_paths,
    totalvi_paths,
]

for tool_label, tool_hyper, tool_paths in tqdm(zip(tools_labels, tools_hyper, tools_paths)):
    for paired_rate, path in zip(paired_rates, tool_paths):
        if path is not None:
            if type(path) == list:
                results = get_scib_results(
                    os.path.join(path[0], "default"), results, batch_key, tool_label, hyper=False
                )
            else:
                results = get_scib_results(
                    os.path.join(path, "default"), results, batch_key, tool_label, hyper=False
                )
            if tool_hyper:
                print(path)
                if type(path) == list:
                    num_hyper_1, path_scib_hyper_1, scib_max1 = get_best_hyper(path[0])
                    print(num_hyper_1)
                    num_hyper_2, path_scib_hyper_2, scib_max2 = get_best_hyper(path[1])
                    print(num_hyper_2)
                    path_scib_hyper = (
                        path_scib_hyper_1 if scib_max1 > scib_max2 else path_scib_hyper_2
                    )
                else:
                    num_hyper, path_scib_hyper, scib_max = get_best_hyper(path)
                    print(num_hyper)
                if path_scib_hyper is not None:
                    results = get_scib_results(
                        path_scib_hyper, results, batch_key, tool_label, hyper=True
                    )


In [ ]:
results.to_csv(os.path.join(_OUTDIR_ABS, f"{dataset_name}_scib_paired_rate_results.csv"), index=False)
print(results.shape)
results.head()

In [ ]:
labels = np.load(
    os.path.join(
        "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
        "default",
        "labels.npy",
    ),
    allow_pickle=True,
)
show_legend(labels)


In [ ]:
labels = np.load(
    os.path.join(
        "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
        "default",
        "labels.npy",
    ),
    allow_pickle=True,
)
show_legend(labels, save_name=f"neurips2021_cite_bmmc_umap_legend_label")
modality = np.load(
    os.path.join(
        "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
        "default",
        "modality.npy",
    ),
    allow_pickle=True,
)
show_legend(modality, save_name=f"neurips2021_cite_bmmc_umap_legend_modality")
if batch_key:
    batch = np.load(
        os.path.join(
            "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
            "default",
            "batch.npy",
        ),
        allow_pickle=True,
    )
    show_legend(batch, save_name=f"neurips2021_cite_bmmc_umap_legend_batch")


In [ ]:
labels = np.load(
    os.path.join(
        "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
        "default",
        "labels.npy",
    ),
    allow_pickle=True,
)
show_legend(labels, save_name=f"neurips2021_cite_bmmc_umap_legend_label")
modality = np.load(
    os.path.join(
        "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
        "default",
        "modality.npy",
    ),
    allow_pickle=True,
)
show_legend(modality, save_name=f"neurips2021_cite_bmmc_umap_legend_modality")
if batch_key:
    batch = np.load(
        os.path.join(
            "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_0.2_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
            "default",
            "batch.npy",
        ),
        allow_pickle=True,
    )
    show_legend(batch, save_name=f"neurips2021_cite_bmmc_umap_legend_batch")


In [ ]:
generate_results_tables(results, save_name=f"{_OUTDIR_ABS}/{dataset_name}")


In [ ]:
results_plot = generate_results_plot(results, paired_rates, tools_labels)


In [ ]:
def plot_performance_vertical(tools_labels, results, results_plot, bar_width=0.05, bar_spacing=0, figsize=(14, 7), show_legend=True, save_name=False, figsize_double=False, figsize_overwrite=None):
    results_plot = results_plot[::-1]
    if figsize_overwrite is not None:
        set_figure_params(figsize_overwrite, 2)
    elif figsize_double:
        set_figure_params('double', 2)
    else:
        set_figure_params('single', 3)

    def lighten_color(color, amount=0.6):
        c = np.array(mcolors.to_rgba(color))
        white = np.array([1, 1, 1, 1])
        return tuple((1 - amount) * c + amount * white)
    groups = [len(tools_labels)] + [len(tools_labels) - 2] * 4
    groups = groups[::-1]
    data = results_plot
    base_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5', '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5']
    biological_preseveration_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    batch_colors = [lighten_color(color, amount=0.6) for color in biological_preseveration_colors]
    modality_colors = [lighten_color(color, amount=0.3) for color in biological_preseveration_colors]
    colors = [biological_preseveration_colors, modality_colors, batch_colors]
    base_colors = biological_preseveration_colors
    fig, ax = plt.subplots()
    indices = np.arange(len(paired_rates))
    group_centers = []
    for i, group_data in enumerate(data):
        group_center = indices[i] + (bar_width + bar_spacing) * (groups[i] - 1) / 2
        group_centers.append(group_center)
        for j, j_pos in zip(range(groups[i]), range(groups[i] - 1, -1, -1)):
            left = np.zeros(group_data.shape[1])
            for k in range(group_data.shape[1]):
                color = colors[k][j]
                if j_pos == groups[i] - 1:
                    bar_position = indices[i] + j_pos * (bar_width + bar_spacing) + bar_width * 0.6
                else:
                    bar_position = indices[i] + j_pos * (bar_width + bar_spacing)
                ax.barh(bar_position, group_data[j, k], bar_width, left=left, color=color, capsize=3, edgecolor='black')
                left += group_data[j, k]
    ax.axhline(y=group_centers[-1] - 0.5, color='black', linestyle='--', linewidth=1)
    ax.set_yticks(group_centers)
    ax.set_yticklabels([f'{pr:.1f}' for pr in paired_rates[::-1]])
    ax.set_ylabel('paired rate')
    ax.set_xlabel('combined scib')
    ax.set_xlim(left=0, right=max(results[results['scib_type'] == 'combined']['scib']) + 0.05)
    legend_handles_tools = [plt.Rectangle((0, 0), 1, 1, color=color, edgecolor='black') for color in base_colors[:len(tools_labels)]]
    scib_legend_handles_tools = [plt.Rectangle((0, 0), 1, 1, color='white', edgecolor='white')] + [plt.Rectangle((0, 0), 1, 1, color=lighten_color(base_colors[0], amount=lighten_amount), edgecolor='black') for lighten_amount in [0, 0.3, 0.6]]
    scib_labels = ['', 'biological preservation', 'modality alignment', 'batch correction']
    if show_legend:
        ax.legend(legend_handles_tools + scib_legend_handles_tools, tools_labels + scib_labels, loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()
plot_performance_vertical(tools_labels, results, results_plot, bar_width=0.1, show_legend=True, figsize_double=False, save_name=f'{_OUTDIR_ABS}/{dataset_name}_vfinal')


In [ ]:
plot_performance(
    tools_labels,
    results,
    results_plot,
    bar_width=0.1,
    show_legend=True,
    save_name=f"{_OUTDIR_ABS}/{dataset_name}",
)


In [ ]:
plot_performance(
    tools_labels,
    results,
    results_plot,
    bar_width=0.1,
    show_legend=False,
    save_name=f"{_OUTDIR_ABS}/{dataset_name}_without_legend",
)


In [ ]:
results_plot = generate_results_plot_old(results, paired_rates, tools_labels)


In [ ]:
results_plot = generate_results_plot(results, paired_rates, tools_labels)


In [ ]:
plot_performance(
    tools_labels,
    results,
    results_plot,
    bar_width=0.1,
    show_legend=True,
    save_name=f"{_OUTDIR_ABS}/{dataset_name}",
)


In [ ]:
plot_performance(
    tools_labels,
    results,
    results_plot,
    bar_width=0.1,
    show_legend=True,
    save_name=f"{_OUTDIR_ABS}/{dataset_name}",
)


In [ ]:
plot_performance(
    tools_labels,
    results,
    results_plot,
    bar_width=0.1,
    show_legend=True,
    save_name=f"{_OUTDIR_ABS}/{dataset_name}",
)


In [ ]:
plot_performance_old(
    tools_labels,
    results,
    results_plot,
    bar_width=0.06,
    show_legend=True,
    save_name=f"{_OUTDIR_ABS}/{dataset_name}",
)


In [ ]:
plot_performance_old(
    tools_labels,
    results,
    results_plot,
    bar_width=0.1,
    show_legend=False,
    save_name=f"{_OUTDIR_ABS}/{dataset_name}",
)


In [ ]:
plot_performance(
    tools_labels,
    results,
    results_plot,
    bar_width=0.1,
    show_legend=False,
    save_name=f"{_OUTDIR_ABS}/{dataset_name}",
)


In [ ]:
plot_performance(tools_labels, results, results_plot, bar_width=0.1, figsize=(10, 5))


In [ ]:
plot_performance(tools_labels, results, results_plot, bar_width=0.1, figsize=(10, 5))
